### Qiskit

In [106]:
import sympy as sp
from sympy.logic import SOPform

# 1. 定义真值表（例如：2输入比特，真值表[1,1,0,1]）
input_bits = 2
truth_table = [1, 1, 0, 1]  # 对应x=00,01,10,11的输出

# 2. 用sympy将真值表转化为逻辑表达式（最小项之和）
variables = [sp.Symbol(f'x{i}') for i in range(input_bits)]
minterms = [i for i, val in enumerate(truth_table) if val == 1]  # f(x)=1的输入索引
expr = SOPform(variables, minterms)  # 生成"与或"表达式

# 3. 转换为Qiskit支持的表达式格式（替换符号为字符串）
logic_expr = str(expr).replace('&', ' & ').replace('|', ' | ').replace('~', '~')
print(logic_expr)
# 4. 生成Oracle

from qiskit.circuit.library import PhaseOracle
from qiskit import QuantumCircuit

oracle = PhaseOracle(logic_expr)

# 转换为"|x⟩|0⟩→|x⟩|f(x)⟩"形式的电路（PhaseOracle默认是相位翻转，需稍作调整）
# 方法：在辅助比特上添加H门，将相位翻转转化为振幅翻转（即目标态翻转）
qc = QuantumCircuit(oracle.num_qubits, name="U_f")
qc.append(oracle.to_gate(), range(oracle.num_qubits))
# 若需要显式的输出比特（而非相位），可对辅助比特应用H门
qc.h(oracle.num_qubits - 1)  # 最后一个比特为辅助比特

qc.draw("mpl")

x1  |  ~x0


MissingOptionalLibraryError: "The 'tweedledum' library is required to use 'PhaseOracle'. You can install it with 'pip install tweedledum'."

### pytket

In [107]:
from pytket import Circuit
from pytket.extensions.bdd import bdd_to_circuit
from dd import BDD

# 用BDD定义函数f(a,b) = a & b
bdd = BDD()
a, b = bdd.add_vars(2)
f = a & b

# 转换为Oracle电路
oracle = bdd_to_circuit(f)
print(oracle)  # 输出优化后的电路

ModuleNotFoundError: No module named 'pytket.extensions'

### Quipper

In [110]:
import Quipper

# -- 定义布尔函数：f(x0,x1) = x0 ∨ ¬x1
f :: [Bool] -> Bool
f [x0, x1] = x0 || not x1

# -- 生成Oracle电路
oracle :: (Qubit, Qubit) -> Qubit -> Circ ()
oracle (x0, x1) target = do
    controlled_not (x0, x1) target  -- 自动优化的多控门

SyntaxError: invalid syntax (1993099679.py, line 4)